
# SPS Program Fit Explorer RAG System

## Task 1 — Domain Selection and Importance

**Domain:** Program Fit Explorer RAG System


**Brief Description:**
Program Fit Explorer RAG System is a retrieval-augmented assistant that answers SPS graduate program and policy questions by searching official SLU catalog/policy PDFs and generating responses grounded in the exact retrieved passages, reducing guesswork and hallucinations compared to a standalone LLM.

**Why it matters:**

The Program Fit Explorer RAG System matters because students and program teams make high-stakes decisions based on program requirements and policy details—things like prerequisites, grading rules, late work penalties, attendance expectations, academic integrity rules, and advising processes. If this information is misunderstood, students can waste time and money, miss critical deadlines, or prepare for the wrong skill set.

**What problems users face without RAG:**

Without RAG, an LLM relies on its internal knowledge and pattern matching, which can lead to confident but incorrect answers. It may mix policies across different universities, programs, or semesters, or invent responses (hallucinations) that sounds right. Users then face problems like receiving the wrong prerequisite guidance, misunderstanding grading or late submission policies, getting vague answers instead of document-specific rules, and having no way to verify the source. This is especially risky when questions require exact wording, exceptions, and up-to-date policy context.

**What questions your RAG system should answer:**

The system should answer practical, document-specific questions such as:

- “What prerequisites or core skills are expected for this program?”

- “What is the grading breakdown and late submission policy for this course?”

- “What does the academic integrity policy allow or prohibit (e.g., collaboration, AI use, citations)?”

- “What are the attendance expectations and consequences for missing class?”

- “Which office handles advising for this program and what is the process to contact them?”

- “What are the program requirements for progression or graduation (credits, GPA, required courses)?”

**Why retrieval-based grounding is required:**

Retrieval-based grounding is required because these answers must be based on authoritative documents (syllabi, handbooks, policy PDFs, official web pages) and often depend on exact wording, dates, or exceptions. Policies also change over time, so relying on a model’s internal memory is not dependable. A retrieval step ensures the assistant pulls the most relevant passages first, then generates an answer that is verifiable, citeable, and consistent with the actual source text—reducing hallucinations and making the system suitable for real student support and program decision-making.

##
Task 2: Build the Knowledge Base & Ingestion Pipeline

In [ ]:
# Installing core libraries for PDF ingestion (PyPDF) and data handling (pandas).
!pip -q install pypdf pandas

In [ ]:
# Download all SLU SPS program/catalog/policy/course PDFs into a local folder so the knowledge base can be rebuilt on every run.

import os, subprocess

RAW_DIR = "/content/sps_rag/raw_pdfs"
os.makedirs(RAW_DIR, exist_ok=True)

PDF_SOURCES = {
    # ---- SPS programs PDFs ----
    "sps_dept_programs.pdf": "https://catalog.slu.edu/colleges-schools/professional-studies/professional-studies.pdf",

    # ---- SPS graduate program PDFs ----
    "sps_program_analytics_ms.pdf": "https://catalog.slu.edu/colleges-schools/professional-studies/analytics-ms/analytics-ms.pdf",
    "sps_program_cybersecurity_ms.pdf": "https://catalog.slu.edu/colleges-schools/professional-studies/cybersecurity-ms/cybersecurity-ms.pdf",
    "sps_program_information_systems_ms.pdf": "https://catalog.slu.edu/colleges-schools/professional-studies/information-systems-ms/information-systems-ms.pdf",
    "sps_program_leadership_org_dev_ma.pdf": "https://catalog.slu.edu/colleges-schools/professional-studies/leadership-organizational-development-ma/leadership-organizational-development-ma.pdf",
    "sps_program_project_management_ms.pdf": "https://catalog.slu.edu/colleges-schools/professional-studies/project-management-ms/project-management-ms.pdf",
    "sps_program_strategic_intelligence_ms.pdf": "https://catalog.slu.edu/colleges-schools/professional-studies/strategic-intelligence-ms/strategic-intelligence-ms.pdf",

    # ---- University policy PDFs ----
    "policy_academic_integrity.pdf": "https://www.slu.edu/provost/policies/academic-and-course/academic-integrity-policy.pdf",
    "policy_grading_system.pdf": "https://catalog.slu.edu/academic-policies/academic-policies-procedures/grading-system/grading-system.pdf",
    "policy_attendance.pdf": "https://catalog.slu.edu/academic-policies/academic-policies-procedures/attendance/attendance.pdf",
    "policy_withdraw_drop.pdf": "https://catalog.slu.edu/academic-policies/academic-policies-procedures/dropping-or-withdrawing-from-courses/dropping-or-withdrawing-from-courses.pdf",
    "policy_incomplete_course.pdf": "https://catalog.slu.edu/academic-policies/academic-policies-procedures/incomplete-course/incomplete-course.pdf",
    "policy_refunds.pdf": "https://catalog.slu.edu/academic-policies/student-financial-services/refunds/refunds.pdf",
    "policy_grade_reports.pdf": "https://catalog.slu.edu/academic-policies/academic-policies-procedures/grade-reports/grade-reports.pdf",

    # ---- Course catalog PDFs (subject-based) ----
    "courses_aa.pdf": "https://catalog.slu.edu/courses-az/aa/aa.pdf",
    "courses_cybr.pdf": "https://catalog.slu.edu/courses-az/cybr/cybr.pdf",
    "courses_is.pdf": "https://catalog.slu.edu/courses-az/is/is.pdf",
    "courses_orld.pdf": "https://catalog.slu.edu/courses-az/orld/orld.pdf",
    "courses_pmgt.pdf": "https://catalog.slu.edu/courses-az/pmgt/pmgt.pdf",
    "courses_intl.pdf": "https://catalog.slu.edu/courses-az/intl/intl.pdf",
}

def download_pdf(url: str, out_path: str):
    # -L follows redirects, -q quiet, --show-progress for progress bar
    subprocess.run(["wget", "-q", "--show-progress", "-O", out_path, url], check=True)

# (Re)download each run to ensure freshness
for fname, url in PDF_SOURCES.items():
    out_path = os.path.join(RAW_DIR, fname)
    download_pdf(url, out_path)
    print("Downloaded:", fname)

print("\nTotal PDFs:", len([f for f in os.listdir(RAW_DIR) if f.lower().endswith(".pdf")]))


In [ ]:
# Ingest PDFs: extract text, clean/normalize it, assign categories, and save a reusable docs.jsonl

import re, json
from datetime import datetime
import pandas as pd
from pypdf import PdfReader

OUT_DIR = "/content/sps_rag/processed"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_JSONL = os.path.join(OUT_DIR, "docs.jsonl")
OUT_STATS = os.path.join(OUT_DIR, "ingestion_stats.csv")

def read_pdf(path: str) -> str:
    reader = PdfReader(path)
    return "\n".join([(p.extract_text() or "") for p in reader.pages])

def normalize_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\u00ad", "")                 # soft hyphen
    text = re.sub(r"-\n(\w)", r"\1", text)            # join hyphenated line breaks
    text = re.sub(r"\n{3,}", "\n\n", text)            # collapse excessive newlines
    text = re.sub(r"[ \t]{2,}", " ", text)            # collapse multiple spaces/tabs
    return text.strip()

def infer_category(filename: str) -> str:
    f = filename.lower()
    if f == "sps_dept_programs.pdf": return "department_catalog"
    if f.startswith("sps_"): return "program_catalog"
    if f.startswith("policy_"): return "policy"
    if f.startswith("courses_"): return "course_catalog"
    return "other"

pdfs = [fn for fn in sorted(os.listdir(RAW_DIR)) if fn.lower().endswith(".pdf")]

docs, stats = [], []
for fn in pdfs:
    path = os.path.join(RAW_DIR, fn)
    raw = read_pdf(path)
    clean = normalize_text(raw)

    docs.append({
        "doc_id": os.path.splitext(fn)[0],
        "filename": fn,
        "source_url": PDF_SOURCES.get(fn, ""),
        "category": infer_category(fn),
        "ingested_at": datetime.utcnow().isoformat(),
        "text": clean
    })
    stats.append({
        "doc_id": os.path.splitext(fn)[0],
        "category": infer_category(fn),
        "n_chars": len(clean),
        "n_words": len(clean.split())
    })

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for d in docs:
        f.write(json.dumps(d, ensure_ascii=False) + "\n")

df_stats = pd.DataFrame(stats).sort_values(["category","n_words"], ascending=[True, False])
df_stats.to_csv(OUT_STATS, index=False)

print("✅ Saved:", OUT_JSONL)
print("✅ Saved:", OUT_STATS)
df_stats.head(20)


In [ ]:
# Sanity Check: print one ingested filename and a text snippet to confirm extraction worked.

print(docs[0]["filename"])
print(docs[0]["text"][:999])


## Task 3 — Implement Multiple Chunking Strategies

In [ ]:
# loads docs.jsonl, create chunks using two strategies (sentence-based and sliding window), and save chunk files + summary stat

# Input:  /content/sps_rag/processed/docs.jsonl  (from Task 2)
# Output: /content/sps_rag/processed/chunks_sentence.jsonl
#         /content/sps_rag/processed/chunks_sliding.jsonl
#         /content/sps_rag/processed/chunking_stats.csv


import os, json, re
from typing import List, Dict, Callable, Any
import pandas as pd

# ----------------------------
# Load docs.jsonl
# ----------------------------
DOCS_JSONL = "/content/sps_rag/processed/docs.jsonl"
assert os.path.exists(DOCS_JSONL), "docs.jsonl not found. Run Task 2 ingestion first."

docs: List[Dict[str, Any]] = []
with open(DOCS_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            docs.append(json.loads(line))

print("Loaded docs:", len(docs))
print("Example doc:", docs[0]["doc_id"], docs[0]["category"], docs[0]["filename"])

OUT_DIR = "/content/sps_rag/processed"
os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------
# 1) Strategy A: Sentence-based chunking
#    - Split into sentences
#    - Merge sentences until ~target_chars
# ----------------------------
def split_into_sentences(text: str) -> List[str]:
    text = (text or "").strip()
    if not text:
        return []
    # basic split on punctuation + whitespace
    sents = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sents if s and s.strip()]

def chunk_by_sentences(text: str, target_chars: int = 900, min_chars: int = 300) -> List[str]:
    sents = split_into_sentences(text)
    chunks = []
    cur = ""

    for s in sents:
        # If adding next sentence exceeds target, finalize current chunk (if big enough)
        if cur and (len(cur) + 1 + len(s) > target_chars):
            if len(cur) >= min_chars:
                chunks.append(cur.strip())
                cur = s
            else:
                # if too small, keep extending (avoid tiny chunks)
                cur = (cur + " " + s).strip()
        else:
            cur = (cur + " " + s).strip() if cur else s

    if cur.strip():
        chunks.append(cur.strip())

    return chunks

# ----------------------------
# 2) Strategy B: Sliding-window chunking
#    - Chunk by words with overlap (token-ish approach)
# ----------------------------
def chunk_sliding_window(text: str, window_words: int = 180, overlap_words: int = 60) -> List[str]:
    words = (text or "").split()
    if not words:
        return []

    chunks = []
    step = max(1, window_words - overlap_words)

    for start in range(0, len(words), step):
        end = start + window_words
        chunk = " ".join(words[start:end]).strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(words):
            break

    return chunks

# ----------------------------
# 3) Apply strategies & save to JSONL
# ----------------------------
def build_chunks(
    docs: List[Dict[str, Any]],
    strategy_name: str,
    chunk_fn: Callable[..., List[str]],
    **kwargs
) -> List[Dict[str, Any]]:
    all_chunks = []
    for d in docs:
        text = d.get("text", "") or ""
        pieces = chunk_fn(text, **kwargs)

        for i, ch in enumerate(pieces):
            all_chunks.append({
                "chunk_id": f"{d['doc_id']}::{strategy_name}::{i}",
                "doc_id": d.get("doc_id", ""),
                "filename": d.get("filename", ""),
                "category": d.get("category", ""),
                "source_url": d.get("source_url", ""),
                "strategy": strategy_name,
                "chunk_index": i,
                "text": ch
            })
    return all_chunks

chunks_sentence = build_chunks(
    docs=docs,
    strategy_name="sentence",
    chunk_fn=chunk_by_sentences,
    target_chars=900,
    min_chars=300
)

chunks_sliding = build_chunks(
    docs=docs,
    strategy_name="sliding",
    chunk_fn=chunk_sliding_window,
    window_words=180,
    overlap_words=60
)

OUT_SENT = os.path.join(OUT_DIR, "chunks_sentence.jsonl")
OUT_SLIDE = os.path.join(OUT_DIR, "chunks_sliding.jsonl")

with open(OUT_SENT, "w", encoding="utf-8") as f:
    for c in chunks_sentence:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

with open(OUT_SLIDE, "w", encoding="utf-8") as f:
    for c in chunks_sliding:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

print("Saved:", OUT_SENT, "| chunks:", len(chunks_sentence))
print("Saved:", OUT_SLIDE, "| chunks:", len(chunks_sliding))

# ----------------------------
# 4) Compare chunk distributions (stats)
# ----------------------------
def summarize_chunks(chunks: List[Dict[str, Any]], name: str) -> Dict[str, Any]:
    word_lens = [len((c.get("text","") or "").split()) for c in chunks]
    char_lens = [len((c.get("text","") or "")) for c in chunks]
    return {
        "strategy": name,
        "num_chunks": len(chunks),
        "avg_words": (sum(word_lens)/len(word_lens)) if word_lens else 0,
        "min_words": min(word_lens) if word_lens else 0,
        "max_words": max(word_lens) if word_lens else 0,
        "avg_chars": (sum(char_lens)/len(char_lens)) if char_lens else 0,
        "min_chars": min(char_lens) if char_lens else 0,
        "max_chars": max(char_lens) if char_lens else 0,
    }

stats_rows = [
    summarize_chunks(chunks_sentence, "sentence"),
    summarize_chunks(chunks_sliding, "sliding"),
]

df_stats = pd.DataFrame(stats_rows)
STATS_PATH = os.path.join(OUT_DIR, "chunking_stats.csv")
df_stats.to_csv(STATS_PATH, index=False)

print("\nChunking stats:")
display(df_stats)
print("Saved:", STATS_PATH)

# ----------------------------
# 5) Side-by-side sample from the same doc (optional)
# ----------------------------
TARGET_DOC_ID = "sps_program_analytics_ms"  # change if you want

sent_samples = [c for c in chunks_sentence if c["doc_id"] == TARGET_DOC_ID][:2]
slide_samples = [c for c in chunks_sliding if c["doc_id"] == TARGET_DOC_ID][:2]

print("\n==============================")
print("SAMPLES for doc_id:", TARGET_DOC_ID)
print("==============================")

print("\n--- Sentence-based sample ---")
for c in sent_samples:
    print("\n", c["chunk_id"])
    print(c["text"][:900])

print("\n--- Sliding-window sample ---")
for c in slide_samples:
    print("\n", c["chunk_id"])
    print(c["text"][:900])


In [ ]:
# Visualizing chunking strategy comparison as a side-by-side bar chart with data labels (e.g., #chunks vs avg words).

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

STATS_CSV = "/content/sps_rag/processed/chunking_stats.csv"
df = pd.read_csv(STATS_CSV).sort_values("strategy")

strategies = df["strategy"].tolist()
num_chunks = df["num_chunks"].to_numpy()
avg_words = df["avg_words"].to_numpy()

x = np.arange(len(strategies))
w = 0.38  # bar width

plt.figure(figsize=(10,4))

bars1 = plt.bar(x - w/2, num_chunks, width=w, color="tab:blue", label="# Chunks")
bars2 = plt.bar(x + w/2, avg_words,  width=w, color="tab:orange", label="Avg Words/Chunk")

plt.title("Chunking Strategy Summary")
plt.xlabel("Chunking strategy")
plt.ylabel("Value")
plt.xticks(x, strategies)
plt.legend()
plt.tight_layout()

# ---- data labels ----
def add_labels(bars, fmt="{:.0f}"):
    for b in bars:
        h = b.get_height()
        plt.text(
            b.get_x() + b.get_width()/2,
            h,
            fmt.format(h),
            ha="center",
            va="bottom",
            fontsize=9
        )

add_labels(bars1, fmt="{:.0f}")   # chunks as integer
add_labels(bars2, fmt="{:.1f}")   # avg words with 1 decimal

plt.show()



### Compare the produced chunks
- **Sentence-based:** 233 chunks, avg **~113 words** (19–301), avg **~805 chars** (212–2196).  
  Chunks follow natural sentence/paragraph boundaries therefore cleaner, self-contained meaning.
- **Sliding window (180 words, 60 overlap):** 220 chunks, avg **~174 words** (57–180), avg **~1235 chars** (361–3254).  
  More uniform size, but boundaries can cut mid-thought (e.g., chunk ends with “As …”), causing mixed/incomplete context.

#### **Best strategy for Program Fit Explorer RAG:**
Sentence-based chunking is the best default because SPS program/policy questions are usually specific (credits, requirements, withdrawal rules), and complete sentences/clauses matter for correct interpretation. Cleaner chunks improve retrieval precision and make grounded answers easier.




## TASK 4 — Embeddings + Retrieval Using NumPy Arrays & FAISS
Steps performed:
1) Load chunks from Task 3
2) Create embeddings using:
   - Open-source: BGE + MiniLM (Used both for comparison)
   - Closed-source: OpenAI embeddings
3) Store embeddings in NumPy arrays
4) Retrieve top-k using:
   - NumPy cosine similarity (required)
   - FAISS IndexFlatIP (optional)
5) Compare retrieval results across models and backends

In [ ]:
# Install libraries for embeddings and retrieval: NumPy, sentence-transformers (BGE/MiniLM), FAISS, and OpenAI SDK.

!pip -q install numpy pandas sentence-transformers faiss-cpu openai

In [ ]:
# Import required libraries needed for embedding generation and FAISS indexing.

import os, json, time
import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer
from openai import OpenAI

In [ ]:
# Loading the chosen chunk file (sentence chunks) and building chunk_texts + chunk_meta structures used by embedding/retrieval

# Choose sentence chunking strategy for Task 4 retrieval
STRATEGY = "sentence"

CHUNKS_PATH = f"/content/sps_rag/processed/chunks_{STRATEGY}.jsonl"
assert os.path.exists(CHUNKS_PATH), f"Missing chunk file: {CHUNKS_PATH} (run Task 3 first)"

chunks = []
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            chunks.append(json.loads(line))

chunk_texts = [c["text"] for c in chunks]
chunk_meta = [{
    "chunk_id": c["chunk_id"],
    "doc_id": c["doc_id"],
    "category": c.get("category",""),
    "filename": c.get("filename",""),
    "source_url": c.get("source_url",""),
    "strategy": c.get("strategy",""),
    "chunk_index": c.get("chunk_index", 0),
} for c in chunks]

print("Loaded chunks:", len(chunks))
print("Example chunk:", chunk_meta[0]["chunk_id"], "|", chunk_meta[0]["doc_id"], "|", chunk_meta[0]["category"])


In [ ]:
# Compute open-source embeddings for all chunks using two models (BGE and MiniLM) and store them in memory.

HF_MODELS = {
    "bge": "BAAI/bge-small-en-v1.5",
    "minilm": "sentence-transformers/all-MiniLM-L6-v2",
}

_st_cache = {}
def get_st(model_name: str) -> SentenceTransformer:
    if model_name not in _st_cache:
        _st_cache[model_name] = SentenceTransformer(model_name)
    return _st_cache[model_name]

def embed_hf_corpus(model_name: str, texts, batch_size=64, normalize=True) -> np.ndarray:
    m = get_st(model_name)
    emb = m.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=normalize
    ).astype(np.float32)
    return emb

emb = {}
for key, name in HF_MODELS.items():
    print("\nEmbedding corpus with:", key, "|", name)
    t0 = time.time()
    emb[key] = embed_hf_corpus(name, chunk_texts, batch_size=64, normalize=True)
    print("Shape:", emb[key].shape, "| time:", round(time.time()-t0, 2), "s")


In [ ]:
# Securely set the OPENAI_API_KEY for this runtime (input at run time; not stored in the notebook).

import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI_API_KEY: ").strip()
print("Key set for this runtime:", "OPENAI_API_KEY" in os.environ)


In [ ]:
# Compute closed-source embeddings (OpenAI text-embedding-3-small) for all chunks and add them alongside open-source embeddings.

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in env before running OpenAI embeddings."
client = OpenAI()

OPENAI_EMBED_MODEL = "text-embedding-3-small"

def embed_openai_corpus(texts, model_name=OPENAI_EMBED_MODEL, batch_size=96, normalize=True) -> np.ndarray:
    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = [t.replace("\n", " ") for t in texts[i:i+batch_size]]
        resp = client.embeddings.create(model=model_name, input=batch)
        vecs.extend([d.embedding for d in resp.data])

    mat = np.array(vecs, dtype=np.float32)
    if normalize:
        mat = mat / (np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12)
    return mat

print("\nEmbedding corpus with: openai |", OPENAI_EMBED_MODEL)
t0 = time.time()
emb["openai"] = embed_openai_corpus(chunk_texts, batch_size=96, normalize=True)
print("Shape:", emb["openai"].shape, "| time:", round(time.time()-t0, 2), "s")


In [ ]:
# Persist embeddings and chunk metadata to disk (.npy + .json) for reproducibility and faster reruns.

OUT_DIR = "/content/sps_rag/processed"
os.makedirs(OUT_DIR, exist_ok=True)

for key in ["bge", "minilm", "openai"]:
    path = os.path.join(OUT_DIR, f"emb_{key}_{STRATEGY}.npy")
    np.save(path, emb[key])
    print("Saved:", path)

# Save metadata for reproducibility
meta_path = os.path.join(OUT_DIR, f"chunk_meta_{STRATEGY}.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(chunk_meta, f, indent=2)
print("Saved:", meta_path)


In [ ]:
# Define manual cosine-similarity retrieval using NumPy (embed query → score all chunks → return top‑k).

def cosine_scores(mat: np.ndarray, q: np.ndarray) -> np.ndarray:
    q = q.astype(np.float32)
    q = q / (np.linalg.norm(q) + 1e-12)
    matn = mat / (np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12)
    return matn @ q

def top_k(scores: np.ndarray, k: int = 5):
    k = min(k, len(scores))
    idx = np.argpartition(scores, -k)[-k:]
    idx = idx[np.argsort(scores[idx])[::-1]]
    return idx, scores[idx]

def embed_query(model_key: str, query: str) -> np.ndarray:
    if model_key in ("bge", "minilm"):
        m = get_st(HF_MODELS[model_key])
        return m.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)[0]
    elif model_key == "openai":
        resp = client.embeddings.create(model=OPENAI_EMBED_MODEL, input=[query.replace("\n"," ")])
        q = np.array(resp.data[0].embedding, dtype=np.float32)
        return q / (np.linalg.norm(q) + 1e-12)
    else:
        raise ValueError("Unknown model_key: " + model_key)

def numpy_retrieve(model_key: str, query: str, k: int = 5):
    q_emb = embed_query(model_key, query)
    scores = cosine_scores(emb[model_key], q_emb)
    idx, sc = top_k(scores, k=k)
    return [{
        "rank": r+1,
        "score": float(sc[r]),
        "chunk_id": chunk_meta[i]["chunk_id"],
        "doc_id": chunk_meta[i]["doc_id"],
        "category": chunk_meta[i]["category"],
        "text": chunk_texts[i],
    } for r, i in enumerate(idx)]


In [ ]:
# Build FAISS indexes for each embedding set and define FAISS-based top‑k retrieval (deployment-scalable alternative to NumPy)

MODEL_KEYS = ["bge", "minilm", "openai"]

faiss_index = {}
for key in MODEL_KEYS:
    mat = emb[key].astype(np.float32)
    D = mat.shape[1]
    index = faiss.IndexFlatIP(D)  # inner product; with normalized vectors, matches cosine ranking
    index.add(mat)
    faiss_index[key] = index
    print("Built FAISS index:", key, "| ntotal:", index.ntotal)

def faiss_retrieve(model_key: str, query: str, k: int = 5):
    q = embed_query(model_key, query).astype(np.float32).reshape(1, -1)
    scores, idx = faiss_index[model_key].search(q, k)
    out = []
    for rank, (i, s) in enumerate(zip(idx[0], scores[0]), start=1):
        out.append({
            "rank": rank,
            "score": float(s),
            "chunk_id": chunk_meta[i]["chunk_id"],
            "doc_id": chunk_meta[i]["doc_id"],
            "category": chunk_meta[i]["category"],
            "text": chunk_texts[i],
        })
    return out


In [ ]:
# Run Task 4 evaluation: compare Top‑k overlaps (NumPy vs FAISS, and model vs model) and print top retrieved docs per query

def ids(res):
    return [r["chunk_id"] for r in res]

def overlap(a, b):
    return len(set(a).intersection(set(b)))

def show_top(res, title):
    print("\n" + title)
    for r in res:
        print(f"[{r['rank']}] score={r['score']:.4f} | {r['doc_id']} | {r['category']}")

TEST_QUERIES = [
    "How many credits are required for the MS in Analytics?",
    "Is the MS in Analytics offered online and what term structure does SPS use?",
    "What is the policy on withdrawing from a course?",
    "What does an incomplete grade mean and what happens next?",
    "Where can I find cybersecurity course descriptions in the catalog?"
]

K = 5

for q in TEST_QUERIES:
    print("\n" + "="*100)
    print("QUERY:", q)

    # NumPy vs FAISS overlap per model
    for mk in MODEL_KEYS:
        rN = numpy_retrieve(mk, q, k=K)
        rF = faiss_retrieve(mk, q, k=K)
        print(f"{mk.upper():6s} | NumPy vs FAISS overlap = {overlap(ids(rN), ids(rF))} / {K}")

    # Model vs Model overlap (NumPy)
    r_bge = numpy_retrieve("bge", q, k=K)
    r_mlm = numpy_retrieve("minilm", q, k=K)
    r_oai = numpy_retrieve("openai", q, k=K)
    print("NumPy overlap | BGE vs MiniLM =", overlap(ids(r_bge), ids(r_mlm)), "/", K)
    print("NumPy overlap | BGE vs OpenAI =", overlap(ids(r_bge), ids(r_oai)), "/", K)
    print("NumPy overlap | MiniLM vs OpenAI =", overlap(ids(r_mlm), ids(r_oai)), "/", K)

    # Print top docs for quick inspection (NumPy)
    show_top(r_bge,  "BGE + NumPy")
    show_top(r_mlm,  "MiniLM + NumPy")
    show_top(r_oai,  "OpenAI + NumPy")


In [ ]:
# Create a compact comparison table summarizing overlaps and top‑1 sources per query/model for reporting/visualization

rows = []
for q in TEST_QUERIES:
    for mk in MODEL_KEYS:
        rN = numpy_retrieve(mk, q, k=K)
        rF = faiss_retrieve(mk, q, k=K)
        rows.append({
            "query": q,
            "model": mk,
            "numpy_vs_faiss_overlap_top5": overlap(ids(rN), ids(rF)),
            "top1_doc_numpy": rN[0]["doc_id"],
            "top1_cat_numpy": rN[0]["category"],
        })

df_comp = pd.DataFrame(rows)
df_comp


From the table, we can see that both NumPy and FAISS gave identical results (overlap=5), meaning the implementation is correct.

For model behavior differences, the top document changes sometimes across embedding models (e.g., MiniLM vs OpenAI on the cybersecurity query), which shows embedding choice affects what evidence the RAG system will retrieve from.

For categories, policy questions correctly map to policy docs; program questions usually map to program_catalog; navigation questions often map to course_catalog or department_catalog, meaning that the retrieval is generally “on-topic.”

### Result Observations:
- **NumPy vs FAISS:** identical Top-5 because both implement cosine-equivalent similarity on normalized vectors. Additionally, FAISS can be seen as a scalability/deployment upgrade.
- **Embedding model choice matters:** BGE and OpenAI tend to retrieve more direct, answer-bearing documents for SPS program/policy queries, whereas MiniLM sometimes retrieves broader contextual docs (department or program overview).
- For the rest of the RAG system, we would recommend using BGE or OpenAI embeddings as the primary retriever, and keep MiniLM as a baseline comparison.
- Across multiple runs, retrieval was largely stable; however, in a few cases the 5th-ranked chunk changed for OpenAI due to near-tie similarity scores around the cutoff. This highlights that rankings can be slightly sensitive at the margin.

## Task 5 - Build the Full RAG Loop


In [ ]:
# Set up Task 5 generation: define prompt templates and a function to format retrieved chunks into a grounded context block

from openai import OpenAI
import os, textwrap

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY first."
gen_client = OpenAI()

GEN_MODEL = "gpt-4o-mini"  # good for demos; swap if needed

def build_context_block(retrieved):
    # retrieved = list of dicts from numpy_retrieve(...) or faiss_retrieve(...)
    blocks = []
    for r in retrieved:
        blocks.append(
            f"[source: {r['doc_id']} | category: {r['category']} | chunk: {r['chunk_id']}]\n{r['text'].strip()}"
        )
    return "\n\n---\n\n".join(blocks)

SYSTEM_RAG = """You are a university/program support assistant for SLU SPS graduate programs.
Answer ONLY using the provided context. If the context does not contain the answer, say:
"I don't have enough information in the provided documents."
When possible, cite sources like: (doc_id). Keep answers concise and factual."""

SYSTEM_NO_RAG = """You are a university/program support assistant.
Answer as best as you can."""


In [ ]:
# Implement the full RAG loop: generate answers with/without RAG, retrieve top‑k evidence, and force grounded responses with citations

def llm_answer(system_prompt, user_prompt, temperature=0.2, max_tokens=350):
    resp = gen_client.chat.completions.create(
        model=GEN_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content.strip()

def answer_without_rag(query):
    return llm_answer(SYSTEM_NO_RAG, query, temperature=0.5)

def answer_with_rag(query, model_key="bge", backend="numpy", k=5):
    if backend == "numpy":
        retrieved = numpy_retrieve(model_key, query, k=k)
    elif backend == "faiss":
        retrieved = faiss_retrieve(model_key, query, k=k)
    else:
        raise ValueError("backend must be 'numpy' or 'faiss'")

    context = build_context_block(retrieved)

    user_prompt = f"""Question:
{query}

Context:
{context}

Instructions:
- Use ONLY the Context.
- Provide a direct answer.
- Add 1-3 citations like (doc_id) where relevant."""
    ans = llm_answer(SYSTEM_RAG, user_prompt, temperature=0.2)
    return ans, retrieved


In [ ]:
# Demo Task 5: run several student queries and print answers with/without RAG plus the top evidence sources used

DEMO_QUERIES = [
    "How many credits are required for the MS in Analytics?",
    "Is the MS in Analytics offered online and what term structure does SPS use?",
    "What is the policy on withdrawing from a course?",
    "What are the admission requirements for MS in Information Systems?",
    "What are the skills needed to excel in the MS in Analytics program?"
]

for q in DEMO_QUERIES:
    print("\n" + "="*110)
    print("QUERY:", q)

    print("\n--- Answer WITHOUT RAG ---")
    no_rag = answer_without_rag(q)
    print(textwrap.fill(no_rag, width=105))

    print("\n--- Answer WITH RAG (BGE + NumPy) ---")
    rag_ans, retrieved = answer_with_rag(q, model_key="bge", backend="numpy", k=5)
    print(textwrap.fill(rag_ans, width=105))

    print("\nTop sources used:")
    for r in retrieved[:3]:
        print(f"- {r['doc_id']} ({r['category']}) | score={r['score']:.4f}")


In [ ]:
# Build a small results table (query, no‑RAG answer, RAG answer, top source) to include in the report/presentation.

rows = []
for q in DEMO_QUERIES:
    no_rag = answer_without_rag(q)
    rag_ans, retrieved = answer_with_rag(q, model_key="bge", backend="numpy", k=5)

    rows.append({
        "query": q,
        "without_rag_answer": no_rag[:250] + ("..." if len(no_rag) > 250 else ""),
        "with_rag_answer": rag_ans[:250] + ("..." if len(rag_ans) > 250 else ""),
        "top1_doc": retrieved[0]["doc_id"],
        "top1_category": retrieved[0]["category"],
    })

df_task5 = pd.DataFrame(rows)
df_task5


In [ ]:
out_csv = f"/content/sps_rag/processed/task5_rag_vs_no_rag_{STRATEGY}.csv"
df_task5.to_csv(out_csv, index=False)
print("Saved:", out_csv)


### Observations:

Across all queries, RAG answers were more specific and SLU/SPS-grounded, while non-RAG answers were generic, not SLU/SPS specific and occasionally guessed details (e.g., “semester-based” structure).

RAG also showed correct “production-style” behavior by refusing to answer when the retrieved context did not contain the needed information like in case of online term structure question.

Policy and admissions questions benefited the most as retrieval consistently pulled the correct policy/program PDFs and produced detailed, citeable responses.

Retrieval scores represent cosine similarity between the query and chunk embeddings, used to rank the most relevant evidence for grounding.

## Task 6 — Analysis & Reflection

For chunking, we compared sentence-based and sliding-window chunking. Sentence chunking produced 233 chunks (avg ~113 words), while sliding windows produced 220 chunks (avg ~174 words, with overlap). In our SPS program/policy domain, many answers appear in short, self-contained passages (credits, delivery format, policy definitions). Sentence chunks had full sentences with context and were more answer-ready, while sliding windows sometimes mixed topics or cut text mid-thought, which can weaken grounding. Overall, sentence chunking fits this domain better, with sliding windows mainly useful when extra surrounding context is needed.

For embeddings + retrieval, we implemented and compared 2 open-source embeddings (BGE and well as MiniLM) and a closed-source embedding (OpenAI text-embedding-3-small), then compared retrieval using NumPy cosine similarity and FAISS. NumPy vs FAISS returned identical Top-5 results (5/5 overlap) as we used normalized vectors and FAISS exact inner-product search (cosine-equivalent ranking). The noticeable differences came from embedding choice: BGE vs MiniLM often had lower overlap, showing that models retrieve from different documents. In our runs, BGE/OpenAI more consistently surfaced direct program/policy passages and were comparatively more accurate, while MiniLM sometimes retrieved broader contextual sources.

In the RAG loop, WITHOUT RAG answers were generic or guessed details that were not specific to SLU or SPS, while RAG answers became SPS-specific and citeable (e.g., correctly stating 30 credits for MS Analytics, and providing detailed SLU withdrawal rules from `policy_withdraw_drop`). A key success was production-style behavior: when documents didn’t contain specific details (e.g., term structure question from the demo queries), the system refused to guess rather than hallucinating. The main limitation was the coverage of knowledge base as missing or unclear facts couldn't be retrieved. To make this production-ready, we’d have to expand the the knowledge base (term structure definitions, key SPS logistics), add metadata-based routing (program vs policy vs course catalog), and maintain a small evaluation set (queries + expected cited answers) to track retrieval quality.


In [ ]:
import os, json, time, re
import numpy as np
import gradio as gr

PROCESSED_DIR = "/content/sps_rag/processed"  # change if needed

OPENAI_CHAT_MODEL = "gpt-4o-mini"
OPENAI_EMBED_MODEL = "text-embedding-3-small"

BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"
MINILM_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

DEMO_QUESTIONS = [
    "How many credits are required for the MS in Analytics?",
    "Is the MS in Analytics offered online and what term structure does SPS use?",
    "What is the policy on withdrawing from a course?",
    "What does an incomplete grade mean and what happens next?",
    "Where can I find cybersecurity course descriptions in the catalog?",
]

# -------------------------
# Utilities
# -------------------------
def l2_normalize(x: np.ndarray) -> np.ndarray:
    if x.ndim == 1:
        return x / (np.linalg.norm(x) + 1e-12)
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-12)

def read_jsonl(path: str):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def cosine_topk(qvec: np.ndarray, mat: np.ndarray, k: int):
    scores = mat @ qvec
    idx = np.argsort(-scores)[:k]
    return idx, scores[idx]

# -------------------------
# Load artifacts (exactly matching your notebook)
# -------------------------
def load_artifacts():
    chunks_sentence_path = os.path.join(PROCESSED_DIR, "chunks_sentence.jsonl")
    chunks_sliding_path  = os.path.join(PROCESSED_DIR, "chunks_sliding.jsonl")

    if not os.path.exists(chunks_sentence_path):
        raise FileNotFoundError(f"Missing: {chunks_sentence_path} (run Task 3)")
    if not os.path.exists(chunks_sliding_path):
        raise FileNotFoundError(f"Missing: {chunks_sliding_path} (run Task 3)")

    chunks_sentence = read_jsonl(chunks_sentence_path)
    chunks_sliding  = read_jsonl(chunks_sliding_path)

    # embeddings: sentence only (your notebook exports these)
    emb_paths = {
        "BGE (open-source)":    os.path.join(PROCESSED_DIR, "emb_bge_sentence.npy"),
        "MiniLM (open-source)": os.path.join(PROCESSED_DIR, "emb_minilm_sentence.npy"),
        "OpenAI (closed-source)": os.path.join(PROCESSED_DIR, "emb_openai_sentence.npy"),
    }

    emb_mats = {}
    for k, p in emb_paths.items():
        if os.path.exists(p):
            emb_mats[k] = l2_normalize(np.load(p).astype(np.float32))

    if not emb_mats:
        raise FileNotFoundError(
            "No sentence embedding matrices found. Expected one of:\n"
            + "\n".join([f"- {v}" for v in emb_paths.values()])
            + "\nRun Task 4 + the save cell (emb_*.npy)."
        )

    return chunks_sentence, chunks_sliding, emb_mats

CHUNKS_SENT, CHUNKS_SLIDE, EMB_MATS = load_artifacts()

# -------------------------
# Embedding (query)
# -------------------------
_st_cache = {}

def get_st_model(name: str):
    from sentence_transformers import SentenceTransformer
    if name not in _st_cache:
        _st_cache[name] = SentenceTransformer(name)
    return _st_cache[name]

def embed_query_open_source(q: str, model_name: str) -> np.ndarray:
    model = get_st_model(model_name)
    v = model.encode([q], normalize_embeddings=True, show_progress_bar=False)
    return np.asarray(v[0], dtype=np.float32)

def get_openai_client():
    from openai import OpenAI
    return OpenAI()

def embed_query_openai(q: str) -> np.ndarray:
    if not os.environ.get("OPENAI_API_KEY", "").strip():
        raise RuntimeError("OPENAI_API_KEY not set (needed for OpenAI embeddings / generation).")
    client = get_openai_client()
    r = client.embeddings.create(model=OPENAI_EMBED_MODEL, input=[q.replace("\n", " ")])
    v = np.asarray(r.data[0].embedding, dtype=np.float32)
    return l2_normalize(v)

# -------------------------
# Prompts + generation
# -------------------------
def build_rag_prompt(query: str, retrieved_chunks: list) -> str:
    ctx = "\n\n".join([f"[{c.get('chunk_id','')}]\n{c.get('text','')}" for c in retrieved_chunks])
    return f"""You are an SLU SPS assistant.
Use ONLY the provided context.
If the answer is not in the context, say: "I don't have enough information in the provided documents."

CONTEXT:
{ctx}

QUESTION:
{query}

Cite chunk_ids in parentheses when possible.
"""

def build_base_prompt(query: str) -> str:
    return f"Answer without using any retrieved documents.\nQuestion: {query}"

def openai_chat(prompt: str) -> str:
    if not os.environ.get("OPENAI_API_KEY", "").strip():
        return "(OPENAI_API_KEY missing) — generation skipped."
    client = get_openai_client()
    r = client.chat.completions.create(
        model=OPENAI_CHAT_MODEL,
        messages=[
            {"role": "system", "content": "Be accurate and follow instructions."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2
    )
    return r.choices[0].message.content

# -------------------------
# Main run
# -------------------------
def run_app(question, mode, embed_backend, chunk_strategy, top_k, min_score):
    try:
        question = (question or "").strip()
        if not question:
            return "", "", [], "", "Enter a question."

        # Choose chunk set
        if chunk_strategy == "Sentence":
            chunks = CHUNKS_SENT
        else:
            # You have sliding chunks, but you do NOT have sliding embeddings saved.
            return (
                "", "", [], "",
                "❌ You selected Sliding chunks, but your notebook did not export emb_*_sliding.npy.\n"
                "Either (1) switch to Sentence, or (2) rerun Task 4 for STRATEGY='sliding' and save emb_*_sliding.npy."
            )

        # Get embedding matrix (sentence only)
        if embed_backend not in EMB_MATS:
            return "", "", [], "", f"❌ Missing embedding file for: {embed_backend}. Check processed/."

        mat = EMB_MATS[embed_backend]

        # Query embedding must match the backend
        if embed_backend == "BGE (open-source)":
            qvec = embed_query_open_source(question, BGE_MODEL_NAME)
            qvec = l2_normalize(qvec)
        elif embed_backend == "MiniLM (open-source)":
            qvec = embed_query_open_source(question, MINILM_MODEL_NAME)
            qvec = l2_normalize(qvec)
        else:
            qvec = embed_query_openai(question)

        # Retrieve
        idx, scores = cosine_topk(qvec, mat, int(top_k))
        idx = [int(i) for i in idx]
        scores = [float(s) for s in scores]
        top_score = scores[0] if scores else 0.0

        retrieved = [chunks[i] for i in idx]

        # Evidence table
        evidence_rows = []
        for r, (c, sc) in enumerate(zip(retrieved, scores), 1):
            txt = c.get("text", "")
            preview = txt[:350].replace("\n", " ") + ("..." if len(txt) > 350 else "")
            evidence_rows.append([r, c.get("chunk_id",""), round(sc,4), preview])

        rag_prompt = build_rag_prompt(question, retrieved)
        base_prompt = build_base_prompt(question)

        with_rag = ""
        without_rag = ""

        if mode in ("Both", "With RAG"):
            if top_score < float(min_score):
                with_rag = "I don't have enough information in the provided documents."
            else:
                with_rag = openai_chat(rag_prompt)

        if mode in ("Both", "Without RAG"):
            without_rag = openai_chat(base_prompt)

        return with_rag, without_rag, evidence_rows, rag_prompt, f"OK | top_score={top_score:.4f}"

    except Exception as e:
        return "", "", [], "", f"❌ {type(e).__name__}: {e}"

# -------------------------
# UI
# -------------------------
with gr.Blocks(title="SLU SPS RAG Assistant — From Scratch") as demo:
    gr.Markdown("# SLU SPS RAG Assistant — From Scratch")

    question = gr.Textbox(label="Question", lines=3)

    demo_radio = gr.Radio(choices=DEMO_QUESTIONS, label="Demo questions (click one to copy)")
    demo_radio.change(fn=lambda x: x or "", inputs=demo_radio, outputs=question)

    with gr.Row():
        mode = gr.Radio(["Both", "With RAG", "Without RAG"], value="Both", label="Mode")
        embed_backend = gr.Radio(
            choices=list(EMB_MATS.keys()),
            value=list(EMB_MATS.keys())[0],
            label="Embeddings (must match saved .npy)"
        )
        chunk_strategy = gr.Radio(["Sentence", "Sliding"], value="Sentence", label="Chunk Strategy")

    with gr.Row():
        top_k = gr.Slider(3, 10, value=5, step=1, label="Top-K")
        min_score = gr.Slider(0.0, 0.6, value=0.2, step=0.05, label="Min similarity threshold")

    run_btn = gr.Button("Run", variant="primary")

    with gr.Row():
        out_rag = gr.Textbox(label="✅ WITH RAG", lines=12)
        out_base = gr.Textbox(label="❌ WITHOUT RAG", lines=12)

    evidence = gr.Dataframe(headers=["Rank", "Chunk ID", "Score", "Preview"], interactive=False)
    rag_prompt_out = gr.Textbox(label="RAG Prompt", lines=8)
    status = gr.Textbox(label="Status", lines=2)

    run_btn.click(
        fn=run_app,
        inputs=[question, mode, embed_backend, chunk_strategy, top_k, min_score],
        outputs=[out_rag, out_base, evidence, rag_prompt_out, status]
    )

# NOTE: Colab Gradio connections can still fail due to proxy/network.
# Best reliability: run locally. If you still try Colab, use share=True.
demo.launch(share=True, debug=True)
